In [1]:
import requests
import time
import os
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
url = "https://openrouter.ai/api/v1/chat/completions"

def ask_llm(user_message, system_message=None, model="nvidia/nemotron-3.5-lightning:free", temperature=0.7):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": user_message})

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        data = response.json()
        return data["choices"][0]["message"]["content"]
    elif response.status_code == 429:
        print("Rate limited, retrying once after 2s...")
        time.sleep(2)
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        print(f"Retry also failed: {response.status_code}")
        return None
    else:
        print(f"Request failed: {response.status_code} - {response.text}")
        return None

# Test it
print(ask_llm("What is 12 * 8?"))
print(ask_llm(
    "Translate 'good morning' to French.",
    system_message="You are a translation assistant. Reply with only the translation, nothing else."
))

Request failed: 401 - {"error":{"message":"API key expired.","code":401,"metadata":{"headers":{"WWW-Authenticate":"Bearer error=\"invalid_token\", error_description=\"API key expired\""}}}}
None
Request failed: 401 - {"error":{"message":"API key expired.","code":401,"metadata":{"headers":{"WWW-Authenticate":"Bearer error=\"invalid_token\", error_description=\"API key expired\""}}}}
None


In [2]:
from pydantic import BaseModel

In [3]:
class SupportTicket(BaseModel):
  customer_name: str
  order_number : int
  issue_description: str

In [4]:
ticket = SupportTicket(
    customer_name="vinay kumar",
    order_number=6748,
    issue_description="Item Dispatched",

)
print(ticket)
print(type(ticket))

customer_name='vinay kumar' order_number=6748 issue_description='Item Dispatched'
<class '__main__.SupportTicket'>


In [5]:
ticket2 = SupportTicket(
    customer_name="anitha",
    order_number="1234",
    issue_description="billing question"
)
print(ticket2.order_number,type(ticket2.order_number))

1234 <class 'int'>


In [6]:
from typing import Literal

In [7]:
class SupportTicketV2(BaseModel):
  customer_name: str
  order_number: int
  issue_description: Literal["SHIPPING","BILLING","DEFECTIVE_ITEM","OTHER"]


In [8]:
ticket3 = SupportTicketV2(
    customer_name="anitha",
    order_number="1234",
    issue_description="BILLING"
)
print(ticket3)

customer_name='anitha' order_number=1234 issue_description='BILLING'


In [16]:
from pydantic import Field

class SupportTicketV3(BaseModel):
  customer_name: str
  order_number: int
  issue_category: Literal["SHIPPING","BILLING","DEFECTIVE_ITEM","OTHER"]
  urgency_score: int = Field(ge=1, le=5, description="urgency from 1(Low) to 5(high)")
  requires_manager: bool = False

In [18]:
t = SupportTicketV3(
    customer_name="preethi",
    order_number=90599,
    issue_category="BILLING",
    urgency_score=4
)
print(t)

customer_name='preethi' order_number=90599 issue_category='BILLING' urgency_score=4 requires_manager=False


In [19]:
json_string = '{"customer_name": "Anita Rao", "order_number": 55210, "issue_category": "SHIPPING", "urgency_score": 2}'

ticket_from_json = SupportTicketV3.model_validate_json(json_string)
print(ticket_from_json)

data_dict = {"customer_name": "Karan Mehta", "order_number": 33012, "issue_category": "OTHER", "urgency_score": 1}
ticket_from_dict = SupportTicketV3.model_validate(data_dict)
print(ticket_from_dict)

customer_name='Anita Rao' order_number=55210 issue_category='SHIPPING' urgency_score=2 requires_manager=False
customer_name='Karan Mehta' order_number=33012 issue_category='OTHER' urgency_score=1 requires_manager=False


In [20]:
print(type(ticket_from_dict))

<class '__main__.SupportTicketV3'>


In [21]:
print(ticket_from_dict.model_dump_json())
print(ticket_from_json.model_dump_json())

{"customer_name":"Karan Mehta","order_number":33012,"issue_category":"OTHER","urgency_score":1,"requires_manager":false}
{"customer_name":"Anita Rao","order_number":55210,"issue_category":"SHIPPING","urgency_score":2,"requires_manager":false}


In [23]:
from pydantic import ValidationError
bad_data ={
    "customer_name":"sanya kapoor",
     "order_number": "not-a-number",
    "issue_category": "DAMAGED",  # not one of our four allowed values
    "urgency_score": 9,
}
try:
  ticket = SupportTicketV3.model_validate(bad_data)
except ValidationError as e:
  print(e)

3 validation errors for SupportTicketV3
order_number
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-a-number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing
issue_category
  Input should be 'SHIPPING', 'BILLING', 'DEFECTIVE_ITEM' or 'OTHER' [type=literal_error, input_value='DAMAGED', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
urgency_score
  Input should be less than or equal to 5 [type=less_than_equal, input_value=9, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
